In [20]:
import os
import random
import shutil

In [21]:
def move_files(img_name, input_folder_img, output_folder_img, input_folder_txt, output_folder_txt):
    lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
    
    # To'liq yo'llarni hosil qilish
    src_i = os.path.join(input_folder_img, img_name)
    dst_i = os.path.join(output_folder_img, img_name)
    src_l = os.path.join(input_folder_txt, lbl_name)
    dst_l = os.path.join(output_folder_txt, lbl_name)

    # Fayllar mavjudligini tekshirib keyin ko'chirish
    if os.path.exists(src_i) and os.path.exists(src_l):
        shutil.move(src_i, dst_i)
        shutil.move(src_l, dst_l)
        moved_files.add(img_name) # Global to'plamga qo'shish
        return True
    return False

def get_classes_from_file(label_path):
    with open(label_path, 'r') as f:
        return [int(line.split()[0]) for line in f.readlines()]

In [22]:
train_img = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/images'
train_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/labels'
valid_image = 'Dataset/Tyre_error_proofing.v15i.yolov9/valid/images'
valid_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/valid/labels/'
test_image = 'Dataset/Tyre_error_proofing.v15i.yolov9/test/images'
test_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/test/labels/'

os.makedirs(os.path.join(valid_image), exist_ok=True)
os.makedirs(os.path.join(valid_lbl), exist_ok=True)
os.makedirs(os.path.join(test_image), exist_ok=True)
os.makedirs(os.path.join(test_lbl), exist_ok=True)

id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(train_lbl):
    print(f"Xato: {train_lbl} papkasi topilmadi!")
else:
    for filename in os.listdir(train_lbl):
        if filename.endswith('.txt'):
            with open(os.path.join(train_lbl, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 14644 ta
black_wheel    : 14434 ta
simple_wheel   : 14118 ta
white_wheel    : 14291 ta


In [28]:
valid_coeff = 0.2
test_coeff = 0.1
versions = ("v0", "v1", "v2", "v3")

train_img = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/images'
train_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/labels'
valid_image = 'Dataset/Tyre_error_proofing.v15i.yolov9/valid/images'
valid_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/valid/labels'
test_image = 'Dataset/Tyre_error_proofing.v15i.yolov9/test/images'
test_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/test/labels'

os.makedirs(os.path.join(valid_image), exist_ok=True)
os.makedirs(os.path.join(valid_lbl), exist_ok=True)
os.makedirs(os.path.join(test_image), exist_ok=True)
os.makedirs(os.path.join(test_lbl), exist_ok=True)

class_map = {
    '80_label': 0,
    'black_wheel': 1,
    'simple_wheel': 2,
    'white_wheel': 3
}

# Kerakli kvotalar
valid_quotas = {
    '80_label': int(count_of_classes["80_label"]*valid_coeff),
    'black_wheel': int(count_of_classes["black_wheel"]*valid_coeff),
    'simple_wheel': int(count_of_classes["simple_wheel"]*valid_coeff),
    'white_wheel': int(count_of_classes["white_wheel"]*valid_coeff)
}

test_quotas = {
    '80_label':     int(count_of_classes["80_label"]*test_coeff),
    'black_wheel':  int(count_of_classes["black_wheel"]*test_coeff),
    'simple_wheel': int(count_of_classes["simple_wheel"]*test_coeff),
    'white_wheel':  int(count_of_classes["white_wheel"]*test_coeff)
}

valid_count_of_classes = {
    '80_label': 0,
    'black_wheel': 0,
    'simple_wheel': 0,
    'white_wheel': 0
}

test_count_of_classes = {
    '80_label': 0,
    'black_wheel': 0,
    'simple_wheel': 0,
    'white_wheel': 0
}

all_images = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.shuffle(all_images)

# Hozirgi hisoblagich
current_counts_of_valid = {k: 0 for k in valid_quotas.keys()}
current_counts_of_test  = {k: 0 for k in test_quotas.keys()}
moved_files = set()

# --- SARALASH LOGIKASI ---

for img in all_images:
    # Agarda rasm allaqachon ko'chirilgan bo'lsa, o'tkazib yuboramiz
    if img in moved_files:
        continue

    # Versiyalarni tekshirish
    if img.rsplit('.', 1)[0].endswith(versions):
        continue

    lbl = img.rsplit('.', 1)[0] + '.txt'
    lbl_path = os.path.join(train_lbl, lbl)
    
    if not os.path.exists(lbl_path):
        continue

    img_classes = get_classes_from_file(lbl_path)
    
    # Har bir shartda "if img not in moved_files" bo'lishi shart emas, 
    # chunki tepada eng birinchi bo'lib tekshirib oldik.

    # 1. 80_label ustuvorligi
    if class_map['80_label'] in img_classes and current_counts_of_test['80_label'] < test_quotas['80_label']:
        if move_files(img, train_img, test_image, train_lbl, test_lbl):
            # Bu rasm ichidagi barcha ob'ektlarni hisobga olamiz
            for c_name, c_id in class_map.items():
                current_counts_of_test[c_name] += img_classes.count(c_id)
            continue
    
    # 2. White_wheel
    if class_map['white_wheel'] in img_classes and current_counts_of_test['white_wheel'] < test_quotas['white_wheel']:
        if move_files(img, train_img, test_image, train_lbl, test_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_test[c_name] += img_classes.count(c_id)
            continue

    # 3. Simple_wheel
    if class_map['simple_wheel'] in img_classes and current_counts_of_test['simple_wheel'] < test_quotas['simple_wheel']:
        if move_files(img, train_img, test_image, train_lbl, test_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_test[c_name] += img_classes.count(c_id)
            continue

    # 4. Black_wheel
    if class_map['black_wheel'] in img_classes and current_counts_of_test['black_wheel'] < test_quotas['black_wheel']:
        if move_files(img, train_img, test_image, train_lbl, test_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_test[c_name] += img_classes.count(c_id)
            continue

print("Tugatildi!")
print("Natijalar:", current_counts_of_test)

Tugatildi!
Natijalar: {'80_label': 1476, 'black_wheel': 1444, 'simple_wheel': 1468, 'white_wheel': 1429}


In [29]:
test_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/test/labels/'

os.makedirs(os.path.join(valid_image), exist_ok=True)
os.makedirs(os.path.join(valid_lbl), exist_ok=True)
os.makedirs(os.path.join(test_image), exist_ok=True)
os.makedirs(os.path.join(test_lbl), exist_ok=True)

id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(test_lbl):
    print(f"Xato: {train_lbl} papkasi topilmadi!")
else:
    for filename in os.listdir(test_lbl):
        if filename.endswith('.txt'):
            with open(os.path.join(test_lbl, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 1476 ta
black_wheel    : 1444 ta
simple_wheel   : 1468 ta
white_wheel    : 1429 ta


In [30]:
for img in all_images:
    # Agarda rasm allaqachon ko'chirilgan bo'lsa, o'tkazib yuboramiz
    if img in moved_files:
        continue

    lbl = img.rsplit('.', 1)[0] + '.txt'
    lbl_path = os.path.join(train_lbl, lbl)
    
    if not os.path.exists(lbl_path):
        continue

    img_classes = get_classes_from_file(lbl_path)

    # 1. 80_label ustuvorligi
    if class_map['80_label'] in img_classes and current_counts_of_valid['80_label'] < valid_quotas['80_label']:
        if move_files(img, train_img, valid_image, train_lbl, valid_lbl):
            # Bu rasm ichidagi barcha ob'ektlarni hisobga olamiz
            for c_name, c_id in class_map.items():
                current_counts_of_valid[c_name] += img_classes.count(c_id)
            continue
    
    # 2. White_wheel
    if class_map['white_wheel'] in img_classes and current_counts_of_valid['white_wheel'] < valid_quotas['white_wheel']:
        if move_files(img, train_img, valid_image, train_lbl, valid_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_valid[c_name] += img_classes.count(c_id)
            continue

    # 3. Simple_wheel
    if class_map['simple_wheel'] in img_classes and current_counts_of_valid['simple_wheel'] < valid_quotas['simple_wheel']:
        if move_files(img, train_img, valid_image, train_lbl, valid_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_valid[c_name] += img_classes.count(c_id)
            continue

    # 4. Black_wheel
    if class_map['black_wheel'] in img_classes and current_counts_of_valid['black_wheel'] < valid_quotas['black_wheel']:
        if move_files(img, train_img, valid_image, train_lbl, valid_lbl):
            for c_name, c_id in class_map.items():
                current_counts_of_valid[c_name] += img_classes.count(c_id)
            continue

print("Tugatildi!")
print("Natijalar:", current_counts_of_valid)

Tugatildi!
Natijalar: {'80_label': 2949, 'black_wheel': 2891, 'simple_wheel': 2824, 'white_wheel': 2858}


In [31]:
valid_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/valid/labels'

os.makedirs(os.path.join(valid_image), exist_ok=True)
os.makedirs(os.path.join(valid_lbl), exist_ok=True)
os.makedirs(os.path.join(test_image), exist_ok=True)
os.makedirs(os.path.join(test_lbl), exist_ok=True)

id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(valid_lbl):
    print(f"Xato: {train_lbl} papkasi topilmadi!")
else:
    for filename in os.listdir(valid_lbl):
        if filename.endswith('.txt'):
            with open(os.path.join(valid_lbl, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 2949 ta
black_wheel    : 2891 ta
simple_wheel   : 2824 ta
white_wheel    : 2858 ta
